In [1]:
import datasets
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer as SummarizationTokenizer
import torch

# Set device (GPU/CPU)
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

Using device: cuda


In [2]:
# Load summarization model and tokenizer (using a smaller model like T5-small or DistilBART for faster processing)
summarization_model_name = 't5-small'  # You can also try 'sshleifer/distilbart-cnn-12-6' for DistilBART
summarizer_tokenizer = SummarizationTokenizer.from_pretrained(summarization_model_name)
summarizer_model = AutoModelForSeq2SeqLM.from_pretrained(summarization_model_name).to(device)

In [3]:
def summarize_premise(premise):
    # T5 requires adding a task prefix like "summarize: " before the input text
    input_text = "summarize: " + premise
    inputs = summarizer_tokenizer(input_text, return_tensors='pt', truncation=True, max_length=512).to(device)
    summary_ids = summarizer_model.generate(inputs['input_ids'], max_length=50, min_length=25, length_penalty=2.0,
                                            num_beams=4, early_stopping=True)
    summary = summarizer_tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary

In [8]:
test = dataset['train'].to_pandas()

In [11]:
test['genre'].value_counts()

telephone     83348
government    77350
travel        77350
fiction       77348
slate         77306
Name: genre, dtype: int64

In [4]:
# Load MultiNLI dataset
dataset = datasets.load_dataset('multi_nli')

# Filter for the 'slate' genre only
train_dataset_slate = dataset['train'].filter(lambda example: example['genre'] == 'slate')

Filter:   0%|          | 0/392702 [00:00<?, ? examples/s]

In [12]:
len(train_dataset_slate)

77306

In [ ]:
# Define the ratio threshold (e.g., premise-to-hypothesis length ratio > 2)
RATIO_THRESHOLD = 2

# Function to calculate length ratio and selectively summarize long premises
def prepare_dataset_nli(examples):
    premises = examples['premise']
    hypotheses = examples['hypothesis']
    
    # Only summarize premises where the ratio is above 2
    summarized_premises = []
    for premise, hypothesis in zip(premises, hypotheses):
        premise_len = len(premise.split())
        hypothesis_len = len(hypothesis.split())
        ratio = premise_len / hypothesis_len if hypothesis_len > 0 else float('inf')
        
        if ratio > RATIO_THRESHOLD:
            summarized_premises.append(summarize_premise(premise))
        else:
            summarized_premises.append(premise)  # Keep original premise if it's short
        
    examples['premise'] = summarized_premises
    return examples

# Apply selective summarization to the filtered 'slate' dataset (this will take time depending on dataset size)
train_dataset_summarized_slate = train_dataset_slate.map(prepare_dataset_nli, batched=True)

# Save summarized dataset to disk for later use
train_dataset_summarized_slate.save_to_disk('./summarized_train_dataset_slate')

Map:   0%|          | 0/77306 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/77306 [00:00<?, ? examples/s]

In [20]:
train_dataset_other_genres = dataset['train'].filter(lambda example: example['genre'] != 'slate')

# Combine the summarized slate data with other genres
combined_train_dataset = datasets.concatenate_datasets([train_dataset_summarized_slate, train_dataset_other_genres])

# Save combined dataset to disk for later use if needed
combined_train_dataset.save_to_disk('./combined_train_dataset')

Saving the dataset (0/1 shards):   0%|          | 0/392702 [00:00<?, ? examples/s]

In [19]:
len(train_dataset_other_genres)

315396

In [21]:
len(combined_train_dataset)

392702